# Lesson 07 - योजना डिजाइन ढाँचा

यस नोटबुकले Microsoft Agent Framework प्रयोग गरी AI एजेन्टहरूका लागि **योजना डिजाइन ढाँचा** देखाउँछ।
तपाईंले जटिल यात्रा अनुरोधहरूलाई संरचित उपकार्यहरूमा विभाजित गर्ने, तीलाई विशेषज्ञ एजेन्टहरूलाई असाइन गर्ने,
र परिणामी योजना सञ्चालन गर्ने तरिका सिक्नुहुनेछ — सबै पायडान्टिक मोडेलहरूले सञ्चालित संरचित आउटपुट प्रयोग गर्दै।


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## कार्य विघटन

कार्य विघटन योजना डिजाइन ढाँचाको मुख्य हो। एकल एजेन्टलाई जटिल अनुरोध अन्त्यदेखि अन्त्यसम्म ह्यान्डल गर्न भन्दाभन्दा, हामी समस्यालाई साना, राम्रोसँग परिभाषित **उपकार्यहरू** मा तोड्छौं। प्रत्येक उपकार्यलाई एक विशेषज्ञ एजेन्ट (जस्तै, उडानहरू, होटलहरू, गतिविधिहरू) लाई स्पष्ट प्राथमिकता र निर्भरता क्रमसँग सुम्पिन्छ।

यस दृष्टिकोणले धेरै फाइदाहरू प्रदान गर्छ:
- **स्पष्टता**: प्रत्येक उपकार्यको एकल जिम्मेवारी हुन्छ।
- **समानान्तरता**: स्वतन्त्र उपकार्यहरू एकै समयमा चल्न सक्छन्।
- **भरपर्दोइ**: असफलता व्यक्तिगत उपकार्यहरूमा सीमित हुन्छ।
- **बजेट ट्रयाकिङ**: लागतहरू प्रति उपकार्य अनुमानित गरिन्छ र जम्मा गरिन्छ।


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## संरचित आउटपुटसहित योजना एजेन्ट सिर्जना गर्ने

योजना एजेन्टले **फ्रन्ट डेस्क समन्वयक**को रूपमा कार्य गर्दछ। उच्च-स्तरीय यात्रा अनुरोध दिइएको अवस्थामा यसले संरचित `TravelPlan` उत्पादन गर्दछ — अनुरोधलाई उपकार्यहरूमा टुक्र्याउने, प्राथमिकताहरू सेट गर्ने, र निर्भरतालाई पहिचान गर्ने ताकि कंसीयर्ज वा कार्यान्वयन तहले काम पूरा गर्न सकोस्।


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## विशेषज्ञ उपकरणहरूसँग योजना कार्यान्वयन गर्दै

एक पटक फ्रन्ट डेस्क एजेन्टले संरचित योजना तयार गरेपछि, **कन्सियर्ज एजेण्ट** त्यसलाई कार्यान्वयन गर्दछ। प्रत्येक विशेषज्ञ उपकरणले एक प्रकारको उपकार्य (उडानहरू, होटलहरू, गतिविधिहरू) व्यवस्थापन गर्दछ। कन्सियर्जले योजनाका उपकार्यहरू निर्भरता अनुक्रममा पुनरावृत्ति गर्दछ र प्रत्येकलाई उपयुक्त उपकरणमा पठाउँछ।


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## सारांश

यस पाठमा तपाईंले AI एजेन्टहरूको लागि **योजना डिजाइन ढाँचा** सिक्नुभयो:

1. **कार्य विभाजन** — एक फ्रन्ट डेस्क योजना एजेन्टले जटिल यात्रा अनुरोधलाई पायडान्टिक मोडेलहरू प्रयोग गरेर संरचित उपकार्यहरूमा विभाजन गर्दछ, प्रत्येकलाई प्राथमिकता र निर्भरताहरू सहित विशेषज्ञ एजेन्टलाई सुम्पन्छ।
2. **संरचित आउटपुट** — `response_format` पास गरेर एजेन्टले स्वतन्त्र पाठको सट्टा मान्य गरिएको `TravelPlan` वस्तु फर्काउँछ, जसले तलको प्रशोधनलाई भरपर्दो बनाउँछ।
3. **योजना कार्यान्वयन** — एक कन्सीयर्ज एजेन्ट उपकार्यहरू मार्फत विशेषज्ञ उपकरणहरू (`book_flight`, `reserve_hotel`, `book_activity`) प्रयोग गरी योजना कार्यान्वयन र परिणाम रिपोर्ट गर्दछ।

यो ढाँचा *के गर्नु पर्छ* (योजना) लाई *कसरी गर्नु पर्छ* (कार्यान्वयन) बाट अलग गर्दछ, जसले एजेन्टहरूलाई बढी मोड्युलर, परीक्षणयोग्य, र विस्तार गर्न सजिलो बनाउँछ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
यो कागजात AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी सटीकताको प्रयास गर्छौं भने पनि, कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटि वा अशुद्धिहरू हुनसक्छन्। मूल कागजात यसको मौलिक भाषामा नै आधिकारिक स्रोत मानिनेछ। महत्वपूर्ण जानकारीको लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट हुने कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
